In [ ]:
import os, glob
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    RocCurveDisplay, PrecisionRecallDisplay
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

import matplotlib.pyplot as plt

#find otomatic csv
candidates = []
for pattern in ["data/**/*final*.csv", "data/**/*merged*.csv", "data/**/*processed*.csv", "data/**/*.csv"]:
    candidates += glob.glob(pattern, recursive=True)

candidates = sorted(list(set(candidates)))
print("Found CSVs:", *candidates[:20], sep="\n- ")

#pick the most logical one
data_path = candidates[0]
df = pd.read_csv(data_path)

print("Using:", data_path)
df.shape, df.head()


In [ ]:
#find status
possible_status_cols = [c for c in df.columns if c.lower() in ["status", "company_status", "state", "outcome"]]
possible_status_cols


In [ ]:
STATUS_COL = possible_status_cols[0]  # gerekirse elle: STATUS_COL = "status"

#definition of success
success_map_1 = {"ipo":1, "acquired":1, "operating":1, "closed":0, "failed":0}

y_raw = df[STATUS_COL].astype(str).str.lower().str.strip()

#unknown labels
y = y_raw.map(success_map_1)

df_ml = df.copy()
df_ml["success"] = y

#hold the ones with label only
df_ml = df_ml.dropna(subset=["success"]).copy()
df_ml["success"] = df_ml["success"].astype(int)

df_ml["success"].value_counts(normalize=True)


In [ ]:
#delete id cols
drop_like = ["id", "uuid", "name", "website", "url"]
drop_cols = [c for c in df_ml.columns if any(k in c.lower() for k in drop_like)]
drop_cols += [STATUS_COL]  # status'u feature olarak kullanmayalım (label leakage)

X = df_ml.drop(columns=drop_cols + ["success"], errors="ignore")
y = df_ml["success"]

#delete empty cols
missing_rate = X.isna().mean()
X = X.loc[:, missing_rate < 0.60]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

X_train.shape, X_test.shape


In [ ]:
num_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols)
    ],
    remainder="drop"
)

len(num_cols), len(cat_cols)


In [ ]:
logreg = Pipeline(steps=[
    ("prep", preprocess),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))
])

logreg.fit(X_train, y_train)
pred = logreg.predict(X_test)
proba = logreg.predict_proba(X_test)[:, 1]

print(classification_report(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, proba))

ConfusionMatrixDisplay(confusion_matrix(y_test, pred)).plot()
plt.show()

RocCurveDisplay.from_predictions(y_test, proba)
plt.show()

PrecisionRecallDisplay.from_predictions(y_test, proba)
plt.show()


In [ ]:
rf = Pipeline(steps=[
    ("prep", preprocess),
    ("model", RandomForestClassifier(
        n_estimators=400,
        random_state=42,
        class_weight="balanced_subsample",
        n_jobs=-1
    ))
])

rf.fit(X_train, y_train)
pred = rf.predict(X_test)
proba = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, proba))


In [ ]:
#featurre names after onehotencoding
ohe = rf.named_steps["prep"].named_transformers_["cat"].named_steps["onehot"]
cat_feature_names = ohe.get_feature_names_out(cat_cols)

feature_names = np.concatenate([num_cols, cat_feature_names])

importances = rf.named_steps["model"].feature_importances_
imp = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(20)

imp.plot(kind="bar")
plt.xticks(rotation=75, ha="right")
plt.tight_layout()
plt.show()

imp


In [ ]:
os.makedirs("figures/ml", exist_ok=True)
os.makedirs("data/predictions", exist_ok=True)

#png olarak kaydet
test_out = pd.DataFrame({"y_true": y_test.values, "y_pred": pred, "y_proba": proba})
test_out.to_csv("data/predictions/test_predictions.csv", index=False)
test_out.head()
